In [1]:
import os, sys
import random
import numpy as np
import pandas as pd
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)
from utils import utils,models

In [2]:
batch_size = 64
model_name = 'ViT-B/16'
device = utils.get_device()
fintuned_model_path = os.path.normpath(os.path.join("..", "artifacts", "vit-b_16_clip_news_finetuned.pth"))
if os.path.exists(fintuned_model_path):
    model, preprocess = utils.load_saved_model(fintuned_model_path, device)
else:
    raise ModuleNotFoundError("Required module not found")
model = model.to(device)
dataset_dir = '../dataset/newsimages_test_and_evaluation_26_v1.0'
images_dir = f'{dataset_dir}/news_images_evaluation'
dataset = pd.read_csv(f"{dataset_dir}/news_articles_evaluation.csv",encoding="latin-1")
dataset.set_index('article_id', inplace=True)
dataset["article_title"] = dataset["article_title"].apply(utils.normalize_text)
records = [
    {
        "image_path": utils.make_path(images_dir, row['image_id']),
        "title": row['article_title']
    }
    for _, row in dataset.iterrows()
]
image_paths = [r["image_path"] for r in records]
model_path = f"../artifacts/{model_name.replace('/','_').lower()}_evaluation_image_embeddings.npy"
if os.path.exists(model_path):
    image_embeddings = np.load(model_path)
else:
    image_embeddings = utils.encode_images(image_paths, model, preprocess, batch_size, device)
    np.save(model_path, image_embeddings)

In [3]:
id_to_index = utils.build_id_to_index(image_paths)
article_titles,ground_truth = utils.build_ground_truth(dataset,images_dir,id_to_index)
data = [(q, img_id[0]) for q, img_id in ground_truth.items()]

In [ ]:
retrieval_clip = models.CLIPRetrieval(model,image_embeddings,preprocess,device)
mrr = utils.compute_mrr(retrieval_clip,data,10)
queries = list(ground_truth.keys())
recalls = utils.evaluate(retrieval_clip,queries,ground_truth)

In [6]:
mrr,recalls

(np.float64(0.169074767300688),
 {1: np.float64(0.10694050991501416),
  5: np.float64(0.251628895184136),
  10: np.float64(0.3356232294617564)})